# **DIAGNOSTIC VERSION - Find Why Neural Reranking Failed**

This notebook will:
1. Test if neural model is actually scoring documents
2. Check if document fetching works
3. Compare scores before/after neural reranking
4. Fix the implementation

## Setup (Same as before)

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

!pip install --upgrade pip -q
!pip install numpy pandas -q
!pip install torch --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers sentence-transformers -q
!pip install pyserini trectools tabulate tqdm -q

print("✅ Installation complete")

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from pyserini.search.lucene import LuceneSearcher
from trectools import TrecQrel, TrecRun, TrecEval
from collections import defaultdict
from tabulate import tabulate
from sentence_transformers import CrossEncoder
import torch

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/TEXT/FinalProject'
QUERIES_PATH = os.path.join(BASE_PATH, 'queriesROBUST.txt')
QRELS_PATH = os.path.join(BASE_PATH, 'qrels_50_Queries')

queries_df = pd.read_csv(QUERIES_PATH, sep='\t', header=None, names=['qid', 'text'], dtype={'qid': str})
TRAIN_QIDS = sorted(queries_df['qid'].tolist(), key=int)[:50]
QUERIES_DICT = dict(zip(queries_df.qid, queries_df.text))

print(f"✅ Loaded {len(TRAIN_QIDS)} queries")

In [ ]:
searcher = LuceneSearcher.from_prebuilt_index('robust04')
searcher.set_bm25(k1=0.6, b=0.4)

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

print("✅ Models loaded")

## DIAGNOSTIC 1: Test Neural Model on Sample Query

In [ ]:
# Take first query as test
test_qid = TRAIN_QIDS[0]
test_query = QUERIES_DICT[test_qid]

print(f"Test Query: {test_query}")
print("\n" + "="*80)

# Get BM25 results
bm25_hits = searcher.search(test_query, k=10)

print("\nBM25 Top-10 Results:")
for i, hit in enumerate(bm25_hits, 1):
    print(f"{i}. DocID: {hit.docid}, Score: {hit.score:.4f}")

# Test neural scoring
print("\n" + "="*80)
print("Testing Neural Reranking...")
print("="*80)

pairs = []
docids = []

for hit in bm25_hits:
    try:
        doc = searcher.doc(hit.docid)
        if doc and doc.raw():
            content = json.loads(doc.raw()).get('contents', '')[:512]
            pairs.append([test_query, content])
            docids.append(hit.docid)
            print(f"✅ Fetched doc {hit.docid}: {len(content)} chars")
    except Exception as e:
        print(f"❌ Failed to fetch {hit.docid}: {e}")

if pairs:
    print(f"\n📊 Scoring {len(pairs)} documents with neural model...")
    neural_scores = cross_encoder.predict(pairs, show_progress_bar=True)
    
    print("\n" + "="*80)
    print("COMPARISON: BM25 vs Neural Scores")
    print("="*80)
    
    comparison = []
    for i, (docid, bm25_hit) in enumerate(zip(docids, bm25_hits[:len(docids)])):
        comparison.append([
            i+1,
            docid,
            f"{bm25_hit.score:.4f}",
            f"{neural_scores[i]:.4f}",
            f"{neural_scores[i] - bm25_hit.score:.4f}"
        ])
    
    print(tabulate(comparison, 
                   headers=["Rank", "DocID", "BM25", "Neural", "Diff"],
                   tablefmt="fancy_grid"))
    
    # Sort by neural scores
    neural_ranked = sorted(zip(docids, neural_scores), key=lambda x: x[1], reverse=True)
    
    print("\n" + "="*80)
    print("RERANKED ORDER (by Neural Scores):")
    print("="*80)
    for i, (docid, score) in enumerate(neural_ranked, 1):
        orig_rank = docids.index(docid) + 1
        movement = orig_rank - i
        arrow = "↑" if movement > 0 else "↓" if movement < 0 else "="
        print(f"{i}. {docid} (was rank {orig_rank}) {arrow} Neural: {score:.4f}")
    
    print("\n" + "="*80)
    if max(neural_scores) - min(neural_scores) > 0.1:
        print("✅ DIAGNOSIS: Neural model IS producing varied scores!")
        print("   Problem is likely in how scores are being integrated.")
    else:
        print("⚠️ DIAGNOSIS: Neural scores have very low variance.")
        print("   Model may not be discriminating well.")
else:
    print("❌ CRITICAL: Could not fetch any documents!")

## DIAGNOSTIC 2: Check Score Integration Bug

In [ ]:
print("Testing score replacement logic...\n")

# Original implementation (BUGGY?)
def neural_rerank_old(query, initial_scores, top_k=100):
    """Original implementation"""
    top_docs = sorted(initial_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    pairs = []
    valid_docids = []
    
    for docid, _ in top_docs:
        try:
            doc = searcher.doc(docid)
            if doc and doc.raw():
                content = json.loads(doc.raw()).get('contents', '')[:512]
                pairs.append([query, content])
                valid_docids.append(docid)
        except:
            continue
    
    if not pairs:
        return initial_scores
    
    neural_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
    
    # THIS IS THE BUG: Replacing with SAME scale scores
    result = initial_scores.copy()
    for docid, neural_score in zip(valid_docids, neural_scores):
        result[docid] = float(neural_score)  # Neural scores are -10 to +10, BM25 is 0-20
    
    return result

# Test on sample
bm25_scores = {h.docid: float(h.score) for h in bm25_hits}
reranked_old = neural_rerank_old(test_query, bm25_scores, top_k=10)

print("BM25 score range:", min(bm25_scores.values()), "to", max(bm25_scores.values()))
print("Neural score range:", min([reranked_old[d] for d in docids]), "to", max([reranked_old[d] for d in docids]))

print("\n⚠️ PROBLEM IDENTIFIED:")
print("   Neural scores (-10 to +10) are much smaller than BM25 scores.")
print("   When we replace top-100, neural docs get LOW scores.")
print("   Bottom 900 docs still have high BM25 scores.")
print("   Result: Neural reranking is IGNORED in final ranking!")

## FIX: Proper Neural Reranking Implementation

In [ ]:
def neural_rerank_FIXED(query, initial_scores, top_k=100):
    """
    FIXED: Properly integrate neural scores.
    
    Strategy: Use neural scores for top-K, heavily downweight rest.
    """
    # Get top-K by BM25
    all_ranked = sorted(initial_scores.items(), key=lambda x: x[1], reverse=True)
    top_docs = all_ranked[:top_k]
    
    # Fetch and score with neural model
    pairs = []
    valid_docids = []
    
    for docid, _ in top_docs:
        try:
            doc = searcher.doc(docid)
            if doc and doc.raw():
                content = json.loads(doc.raw()).get('contents', '')[:512]
                pairs.append([query, content])
                valid_docids.append(docid)
        except:
            continue
    
    if not pairs:
        return initial_scores
    
    # Get neural scores
    raw_neural_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
    
    # CRITICAL FIX: Scale neural scores to be HIGHER than BM25
    max_bm25 = max(initial_scores.values())
    
    # Normalize neural scores to 0-1
    min_neural = min(raw_neural_scores)
    max_neural = max(raw_neural_scores)
    neural_range = max_neural - min_neural
    
    if neural_range > 0:
        normalized_neural = [(s - min_neural) / neural_range for s in raw_neural_scores]
    else:
        normalized_neural = [0.5] * len(raw_neural_scores)
    
    # Scale to be ABOVE all BM25 scores
    # Top neural doc gets max_bm25 * 2
    # Bottom neural doc gets max_bm25 * 1.1
    scaled_neural_scores = [max_bm25 * (1.1 + 0.9 * norm) for norm in normalized_neural]
    
    # Create result
    result = {}
    
    # Neural-scored docs get high scores
    for docid, score in zip(valid_docids, scaled_neural_scores):
        result[docid] = score
    
    # Remaining docs keep BM25 scores (will be ranked below)
    for docid, score in initial_scores.items():
        if docid not in result:
            result[docid] = score
    
    return result

print("✅ Fixed neural reranking function created")

## Test Fixed Version

In [ ]:
print("Testing FIXED neural reranking...\n")

# Get more docs for better test
bm25_hits_100 = searcher.search(test_query, k=100)
bm25_scores_100 = {h.docid: float(h.score) for h in bm25_hits_100}

# Apply fixed reranking
reranked_fixed = neural_rerank_FIXED(test_query, bm25_scores_100, top_k=100)

# Compare top-10
bm25_top10 = [h.docid for h in bm25_hits_100[:10]]
neural_top10 = sorted(reranked_fixed.items(), key=lambda x: x[1], reverse=True)[:10]
neural_top10_ids = [d for d, s in neural_top10]

print("="*80)
print("COMPARISON: BM25 vs FIXED Neural Reranking")
print("="*80)

comparison = []
for i in range(10):
    bm25_doc = bm25_top10[i]
    neural_doc = neural_top10_ids[i]
    
    same = "✓" if bm25_doc == neural_doc else "✗"
    
    # Find where neural doc was in BM25 ranking
    try:
        neural_orig_rank = bm25_top10.index(neural_doc) + 1
        movement = neural_orig_rank - (i + 1)
    except ValueError:
        neural_orig_rank = ">10"
        movement = "N/A"
    
    comparison.append([
        i+1,
        bm25_doc,
        neural_doc,
        same,
        f"was #{neural_orig_rank}",
        movement if movement != "N/A" else "NEW"
    ])

print(tabulate(comparison,
               headers=["Rank", "BM25 Doc", "Neural Doc", "Same?", "Neural Origin", "Movement"],
               tablefmt="fancy_grid"))

# Calculate ranking difference
same_count = sum(1 for row in comparison if row[3] == "✓")
print(f"\n📊 {same_count}/10 documents remain in same position")
print(f"   {10-same_count}/10 documents reranked by neural model")

if same_count < 10:
    print("\n✅ SUCCESS: Neural reranking IS changing the ranking!")
else:
    print("\n⚠️ Neural model produces same ranking as BM25")

## Now Run Full Evaluation with Fixed Neural Reranking

In [ ]:
# Helper functions
def get_bm25_scores(query, k=1000):
    hits = searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}

def get_rm3_scores(query, k=1000):
    searcher.set_rm3(fb_docs=10, fb_terms=10, original_query_weight=0.5)
    hits = searcher.search(query, k=k)
    searcher.unset_rm3()
    return {h.docid: float(h.score) for h in hits}

def evaluate_pipeline(pipeline_func, pipeline_name, queries_dict, qrels):
    all_rows = []
    
    for qid, query_text in tqdm(queries_dict.items(), desc=f"Evaluating {pipeline_name}"):
        try:
            scores = pipeline_func(query_text)
            ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:1000]
            
            for rank, (docid, score) in enumerate(ranked, start=1):
                all_rows.append([qid, "Q0", docid, rank, score, pipeline_name])
        except Exception as e:
            print(f"Error on query {qid}: {e}")
            continue
    
    run_file = f"{pipeline_name}_run.txt"
    pd.DataFrame(all_rows).to_csv(run_file, sep=' ', header=False, index=False)
    run = TrecRun(run_file)
    return TrecEval(run, qrels)

print("✅ Helper functions ready")

In [ ]:
qrels = TrecQrel(QRELS_PATH)

print("\n" + "="*80)
print("EVALUATION WITH FIXED NEURAL RERANKING")
print("="*80)

pipelines = [
    (lambda q: get_bm25_scores(q, k=1000), "1_Baseline_BM25"),
    (lambda q: get_rm3_scores(q, k=1000), "2_BM25_RM3"),
    (lambda q: neural_rerank_FIXED(q, get_bm25_scores(q, k=1000), top_k=100), "3_BM25_Neural_FIXED"),
    (lambda q: neural_rerank_FIXED(q, get_rm3_scores(q, k=1000), top_k=100), "4_RM3_Neural_FIXED"),
]

results = []

for pipeline_func, pipeline_name in pipelines:
    print(f"\n{'='*80}")
    print(f"Testing: {pipeline_name}")
    print("="*80)
    
    te = evaluate_pipeline(pipeline_func, pipeline_name, QUERIES_DICT, qrels)
    
    map_score = te.get_map()
    p5 = te.get_precision(depth=5)
    p10 = te.get_precision(depth=10)
    
    results.append([pipeline_name, map_score, p5, p10])
    
    print(f"MAP: {map_score:.4f}")
    print(f"P@5: {p5:.4f}")
    print(f"P@10: {p10:.4f}")

# Display comparison
print("\n" + "="*80)
print("RESULTS WITH FIXED NEURAL RERANKING")
print("="*80)
headers = ["Pipeline", "MAP", "P@5", "P@10"]
print(tabulate(results, headers=headers, tablefmt="fancy_grid", floatfmt=".4f"))

baseline_map = results[0][1]
print("\n" + "="*80)
print("IMPROVEMENT OVER BASELINE")
print("="*80)
improvements = []
for name, map_score, _, _ in results[1:]:
    improvement = ((map_score - baseline_map) / baseline_map) * 100
    improvements.append([name, f"{improvement:+.2f}%"])

print(tabulate(improvements, headers=["Pipeline", "MAP Gain"], tablefmt="fancy_grid"))

best_idx = max(range(len(results)), key=lambda i: results[i][1])
print(f"\n🏆 BEST PIPELINE: {results[best_idx][0]}")
print(f"   MAP: {results[best_idx][1]:.4f} ({((results[best_idx][1] - baseline_map) / baseline_map * 100):+.2f}% vs baseline)")
print("="*80)